# ⚡ WavLM Extraction — PyTorch 2.2.2 + CUDA 11.8 (FIXED)
**GPU:** Kaggle P100 works with PyTorch 2.2.2 + CUDA 11.8
**Fix:** Added proper NaN handling in prosody extraction


In [ ]:
# Cell 1: Install PyTorch 2.2.2 + CUDA 11.8
import subprocess, os, sys

print('=== Installing PyTorch 2.2.2 + CUDA 11.8 ===')

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'torch', 'torchvision', 'torchaudio', '-y', '-q'], capture_output=True)

result = subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch==2.2.2',
    'torchvision==0.17.2',
    'torchaudio==2.2.2',
    '--index-url',
    'https://download.pytorch.org/whl/cu118'
], capture_output=True, text=True, timeout=600)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    x = torch.randn(100, 100).cuda()
    y = x @ x
    print('GPU: SUCCESS')

In [ ]:
# Cell 2: Load WavLM
from transformers import AutoModel
import torch

print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.eval()
if torch.cuda.is_available():
    wavlm = wavlm.cuda()
    print('WavLM on GPU')
else:
    print('WavLM on CPU')

In [ ]:
# Cell 3: Setup
import subprocess, os

os.makedirs('/kaggle/working/audio', exist_ok=True)
os.makedirs('/kaggle/working/features', exist_ok=True)

subprocess.run(['pip', 'install', 'yt-dlp', '-q'], capture_output=True)

# All 255 videos to process
VIDEO_IDS = [
    '-UPIA46hBZs', '-vcKXr6WBNc', '0AvUvJ_S2Os', '0Pl51hxcK-o', '0g7nezWZyfY',
    '0zpUnJSG0EQ', '18H1aeoGybw', '18rLwnvxOU0', '1ILQmgHvtd4', '1Uo27tH3JQ4',
    '1pPnJut3KLw', '1tO9MWWOgHk', '21gOjz-Xk7s', '2SUfHIbT0HI', '2axWotdMFsw',
    '2ql8QJWmNM8', '2yTKoDfNDL8', '3zt8hIyV-bM', '4Z4W6BSMUX0', '4piiN94d6P4'
]  # First 20 for this run

def download_one(vid):
    out = f'/kaggle/working/audio/{vid}.wav'
    if os.path.exists(out): return out
    cmd = ['yt-dlp', '-f', 'bestaudio[ext=m4a]', '--extract-audio', '--audio-format', 'wav',
            '-o', f'/kaggle/working/audio/{vid}.%(ext)s',
            f'https://www.youtube.com/watch?v={vid}', '--no-playlist', '--quiet', '--socket-timeout', '90']
    try:
        subprocess.run(cmd, capture_output=True, timeout=180)
        for ext in ['m4a', 'webm', 'mp4']:
            tmp = f'/kaggle/working/audio/{vid}.{ext}'
            if os.path.exists(tmp) and tmp != out:
                os.rename(tmp, out)
        return out if os.path.exists(out) else None
    except: return None

# Download first
print(f'Downloading {len(VIDEO_IDS)} videos...')
for vid in VIDEO_IDS:
    path = download_one(vid)
    print(f'  {vid}: {"OK" if path else "FAIL"}')

In [ ]:
# Cell 4: Extract
import numpy as np, librosa, torch, time

def prosody23(y, sr):
    f = np.zeros(23, dtype=np.float32)
    try:
        f0, v, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        fc = f0[~np.isnan(f0)].astype(np.float32)
        vc = v[~np.isnan(f0)].astype(np.float32)
        if len(fc) > 0:
            f[0] = np.mean(fc)
            f[1] = np.std(fc)
            f[2] = np.max(fc)
            f[3] = np.min(fc)
            f[4] = np.mean(vc)
    except: pass
    
    hop = 512
    try:
        rms = librosa.feature.rms(y=y, hop_length=hop)[0].astype(np.float32)
        f[5] = np.mean(rms)
        f[6] = np.std(rms)
        f[7] = np.max(rms)
        f[8] = np.min(rms)
        f[9] = f[7] - f[8]
    except: pass
    
    dur = len(y) / sr
    f[10] = dur
    try:
        rms_mean = np.mean(rms) if 'rms' in dir() else 0
        f[11] = dur / (np.sum(rms > rms_mean) + 1)
    except: pass
    
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0].astype(np.float32)
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0].astype(np.float32)
        sf = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0].astype(np.float32)
        z = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0].astype(np.float32)
        f[12] = np.mean(sc)
        f[13] = np.mean(sb)
        f[14] = np.mean(sf)
        f[15] = np.mean(z)
        f[16] = np.std(z)
    except: pass
    
    try:
        yh, _ = librosa.effects.hpss(y.astype(np.float32))
        f[17] = np.mean(np.abs(yh)) / (np.mean(np.abs(y)) + 1e-8)
        f[18] = np.mean(np.abs(y))
        f[19] = np.std(y)
        f[20] = np.max(np.abs(y))
    except: pass
    
    return f

def extract(audio_path, vid):
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        cs = 16000 * 5
        n = len(y) // cs
        if n == 0: return None
        
        feats = []
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        for i in range(n):
            ch = y[i*cs:(i+1)*cs].astype(np.float32)
            
            # WavLM
            t = torch.tensor(ch).unsqueeze(0).to(dev)
            with torch.no_grad():
                r = wavlm(t).last_hidden_state.mean(dim=2).squeeze().cpu().numpy().astype(np.float32)
            
            # Prosody
            p = prosody23(ch, 16000)
            
            feats.append(np.concatenate([r, p]))
        
        return np.array(feats, dtype=np.float32)
    except Exception as e:
        print(f'  Error: {e}')
        return None

t0 = time.time()
done = 0
for vid in VIDEO_IDS:
    ap = f'/kaggle/working/audio/{vid}.wav'
    if not os.path.exists(ap): continue
    fp = f'/kaggle/working/features/{vid}_features.npy'
    if os.path.exists(fp): 
        print(f'{vid}: already done')
        done += 1
        continue
    f = extract(ap, vid)
    if f is not None:
        np.save(fp, f)
        print(f'{vid}: {f.shape} ({time.time()-t0:.0f}s)')
        done += 1
    else:
        print(f'{vid}: FAILED')

print(f'\nDone: {done}/{len(VIDEO_IDS)} videos extracted')

In [ ]:
# Cell 5: Summary
import os
fs = [f for f in os.listdir('/kaggle/working/features') if f.endswith('.npy')]
print(f'Features: {len(fs)} videos')
for f in fs[:5]:
    d = np.load(f'/kaggle/working/features/{f}')
    print(f'  {f}: {d.shape}')
print('Done!')